In [1]:
# Prototype code to parse a case from text to dataclass

import json

# This is MSD #1. Currently parsing a case is nontrivial, need to make it much faster later
hopc = """A 26-year-old woman comes to the office because of a 3-day history of lower abdominal pain. She is 18 weeks pregnant by dates. The patient describes the pain as sharp, steady, and radiating across her lower abdomen bilaterally. Last night she developed new nausea and vomiting. She has not been able to keep down any food or drink this morning. She had a normal bowel movement yesterday. She says she felt cold and shivering this morning, followed by feeling warm; however, she did not check her temperature. She denies vaginal bleeding."""

systems_review = """General: Patient feels generally weak and ill but was in her usual state of health until 3 days ago. She has gained approximately 5 lbs (2.3 kg) in the pregnancy so far.

Skin: She denies rash.

HEENT: Her mouth feels dry. No headache, nasal congestion, or sore throat.

Pulmonary: She denies cough or shortness of breath.

Cardiovascular: She denies chest pain or palpitations.

Gastrointestinal: She has had a decreased appetite for 1 day and has been unable to keep any food or drink down this morning due to nausea and vomiting. She has not had diarrhea or constipation.

Genitourinary: She reports a frequent urge to urinate and a sensation of incomplete bladder emptying for the past 3 days. No dysuria or hematuria. She is G1P0A0 and has been seeing an obstetrician for all routine visits and testing. No vaginal bleeding.

Musculoskeletal: She reports mild diffuse low back pain. No generalized muscle aches.

Neurologic: Noncontributory

Psychiatric: Noncontributory"""

pmh = """Medical history: Mild intermittent asthma diagnosed in childhood requiring only occasional rescue inhaler use, no hospitalizations for asthma. She is otherwise healthy.

Surgical history: Wisdom teeth removed at age 18.

Medications: Albuterol inhaler as needed, about once a month. Daily prenatal vitamin.

Allergies: No known drug allergies.

Family history: Mother is healthy at age 50. Father is 53 years old with high blood pressure.

Social history: Patient is employed as an engineer. She exercises 3 days/week. She drinks 2 glasses of wine per week but stopped when she found out she was pregnant. She does not smoke or use any illicit drugs. She has not had any recent travel. She is monogamous with 1 male partner."""

physical_examination = """Physical Examination
General appearance: Well-developed female, appears tired and ill but in no apparent distress.

Vital signs:
Temperature: 38.8° C
Pulse: 120 beats/minute
BP: 110/76 mm Hg
Respirations: 20/minute

Skin: Hot, diaphoretic. No rash or cyanosis.

HEENT: Dry mucous membranes, no scleral icterus or conjunctival injection, neck is supple, no adenopathy.

Pulmonary: Breath sounds are equal bilaterally with good air movement in all fields. There are faint inspiratory crackles present on the left side heard more at the base. No wheezing.

Cardiovascular: Mild tachycardia with regular rhythm; no murmurs, rubs, or gallop. No peripheral edema.

Gastrointestinal: Bowel sounds normal. Abdomen soft, diffusely tender across the lower abdomen bilaterally with no guarding, rigidity, or rebound. No tenderness in the upper quadrants, gravid, non-tender uterus 2 cm below the level of the umbilicus. No inguinal or femoral hernias. Rectal examination non-tender, stool brown, heme negative.

Genitourinary: Normal external genitalia, no cervical motion tenderness, scant thin white discharge from the cervical os, which appears closed. Fetal heartbeat regular at 150 beats/minute.

Musculoskeletal: No swelling, tenderness or deformity of joints or extremities. Positive costovertebral angle tenderness bilaterally.

Neurologic: Unremarkable

Mental status: Alert and oriented, with fluent and coherent speech."""

primary_differentials = """appendicitis: Appendicitis may cause lower abdominal pain, nausea, and vomiting.
pelvic inflammatory disease: Although uncommon, PID may occur during pregnancy and cause significant morbidity.
pyelonephritis: Pyelonephritis can cause fever and urinary frequency.
sepsis: Patient has significant fever, tachycardia, and systemic symptoms, and she should be evaluated for sepsis.
viral infection: Numerous viral infections can cause fever, malaise, vomiting, and abdominal discomfort."""

secondary_differentials = """pyelonephritis: Patient has systemic manifestations including costovertebral tenderness accompanied by indications of infection in the urine.
sepsis: Patient appears ill and has fever, tachycardia, tachypnea, an elevated WBC count, and an apparent source of infection."""

diagnosis = "pyelonephritis and sepsis"

management = """admission: Systemic bacterial infection is a threat to the pregnancy. This patient needs IV antibiotics and fluids until her infection is controlled and she is able to tolerate oral fluids and drugs.
iv fluids: The patient needs IV fluids beyond the maintenance rate.
ceftriaxone 1g IV daily: A broad-spectrum cephalosporin is an acceptable initial treatment of pyelonephritis and possible sepsis during pregnancy. Cefazolin and cefepime are also acceptable antibiotics."""

In [2]:
json.dumps({"hopc": hopc,
           "systems_review": systems_review,
           "pmh": pmh,
           "physical_examination": physical_examination,
           "primary_differentials": primary_differentials,
           "secondary_differentails": secondary_differentials,
           "diagnosis": diagnosis,
           "management": management})

'{"hopc": "A 26-year-old woman comes to the office because of a 3-day history of lower abdominal pain. She is 18 weeks pregnant by dates. The patient describes the pain as sharp, steady, and radiating across her lower abdomen bilaterally. Last night she developed new nausea and vomiting. She has not been able to keep down any food or drink this morning. She had a normal bowel movement yesterday. She says she felt cold and shivering this morning, followed by feeling warm; however, she did not check her temperature. She denies vaginal bleeding.", "systems_review": "General: Patient feels generally weak and ill but was in her usual state of health until 3 days ago. She has gained approximately 5 lbs (2.3 kg) in the pregnancy so far.\\n\\nSkin: She denies rash.\\n\\nHEENT: Her mouth feels dry. No headache, nasal congestion, or sore throat.\\n\\nPulmonary: She denies cough or shortness of breath.\\n\\nCardiovascular: She denies chest pain or palpitations.\\n\\nGastrointestinal: She has had 

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
%env OPENAI_API_KEY=""
%env HUGGINGFACE_TOKEN = ""
                   
if not 'dirguard' in locals():
  %cd ..
  dirguard = True

%aimport vivabench
%aimport vivabench.util
%aimport vivabench.ontology.concepts

from vivabench.ontology import concepts
from vivabench.ontology.concepts import *

env: OPENAI_API_KEY=""
env: HUGGINGFACE_TOKEN=""
/mnt/c/Users/Chris/vivabench


In [4]:
hopc_parsed = HistoryOfPresentingComplaint(
    brief_history="A 26-year-old woman comes to the office because of a 3-day history of lower abdominal pain. She is 18 weeks pregnant by dates.",
    full_history=hopc
)

In [6]:
# Systems is relatively easy to parse
_systems = {k.lower(): v for k, v in [t.split(": ") for t in systems_review.split("\n\n")]}
systems = SystemsReview.from_dict(_systems)
print(systems.prompt())

GENERAL: Patient feels generally weak and ill but was in her usual state of health until 3 days ago. She has gained approximately 5 lbs (2.3 kg) in the pregnancy so far.
SKIN: She denies rash.
HEENT: Her mouth feels dry. No headache, nasal congestion, or sore throat.
PULMONARY: She denies cough or shortness of breath.
CARDIOVASCULAR: She denies chest pain or palpitations.
GASTROINTESTINAL: She has had a decreased appetite for 1 day and has been unable to keep any food or drink down this morning due to nausea and vomiting. She has not had diarrhea or constipation.
GENITOURINARY: She reports a frequent urge to urinate and a sensation of incomplete bladder emptying for the past 3 days. No dysuria or hematuria. She is G1P0A0 and has been seeing an obstetrician for all routine visits and testing. No vaginal bleeding.
MUSCULOSKELETAL: She reports mild diffuse low back pain. No generalized muscle aches.
NEUROLOGIC: Noncontributory
PSYCHIATRIC: Noncontributory



In [7]:
_pmh = {k.lower().replace(" ", "_"): v for k, v in [t.split(": ") for t in pmh.split("\n\n")]}
pmh_parsed = PastMedicalHistory.from_dict(_pmh)
print(pmh_parsed.prompt())

MEDICAL HISTORY: Mild intermittent asthma diagnosed in childhood requiring only occasional rescue inhaler use, no hospitalizations for asthma. She is otherwise healthy.
SURGICAL HISTORY: Wisdom teeth removed at age 18.
MEDICATIONS: Albuterol inhaler as needed, about once a month. Daily prenatal vitamin.
ALLERGIES: No known drug allergies.
FAMILY HISTORY: Mother is healthy at age 50. Father is 53 years old with high blood pressure.
SOCIAL HISTORY: Patient is employed as an engineer. She exercises 3 days/week. She drinks 2 glasses of wine per week but stopped when she found out she was pregnant. She does not smoke or use any illicit drugs. She has not had any recent travel. She is monogamous with 1 male partner.



In [8]:
pe_dict = {}
_pe = physical_examination.split("\n\n")
pe_dict['general'] = _pe[0].split(": ")[-1]

for t in _pe[2:]:
    k, v = t.split(": ")
    pe_dict[k.lower().replace(" ", "_")] = v

# Vitals are wrong here also - need to fix !!!
pe = PhysicalExamination.from_dict(pe_dict)
print(pe.prompt())

VITALS
- Temperature: 36.5 °C
- Pulse: 60 beats/minute
- Blood Pressure: 110/70 mm Hg
- Respiratory Rate: 12 / minute

GENERAL: Well-developed female, appears tired and ill but in no apparent distress.
SKIN: Hot, diaphoretic. No rash or cyanosis.
HEENT: Dry mucous membranes, no scleral icterus or conjunctival injection, neck is supple, no adenopathy.
PULMONARY: Breath sounds are equal bilaterally with good air movement in all fields. There are faint inspiratory crackles present on the left side heard more at the base. No wheezing.
CARDIOVASCULAR: Mild tachycardia with regular rhythm; no murmurs, rubs, or gallop. No peripheral edema.
GASTROINTESTINAL: Bowel sounds normal. Abdomen soft, diffusely tender across the lower abdomen bilaterally with no guarding, rigidity, or rebound. No tenderness in the upper quadrants, gravid, non-tender uterus 2 cm below the level of the umbilicus. No inguinal or femoral hernias. Rectal examination non-tender, stool brown, heme negative.
GENITOURINARY: Nor

In [9]:
primary_ddx = DifferentialList([Differential(name=k, reason=v) for k, v in [t.split(": ") for t in primary_differentials.split("\n")]])

print(primary_ddx.prompt(full_ddx=True))

- Appendicitis
>> Appendicitis may cause lower abdominal pain, nausea, and vomiting.
- Pelvic inflammatory disease
>> Although uncommon, PID may occur during pregnancy and cause significant morbidity.
- Pyelonephritis
>> Pyelonephritis can cause fever and urinary frequency.
- Sepsis
>> Patient has significant fever, tachycardia, and systemic symptoms, and she should be evaluated for sepsis.
- Viral infection
>> Numerous viral infections can cause fever, malaise, vomiting, and abdominal discomfort.




In [10]:
secondary_ddx = DifferentialList([Differential(name=k, reason=v) for k, v in [t.split(": ") for t in secondary_differentials.split("\n")]])
print(secondary_ddx.prompt(full_ddx=True))

- Pyelonephritis
>> Patient has systemic manifestations including costovertebral tenderness accompanied by indications of infection in the urine.
- Sepsis
>> Patient appears ill and has fever, tachycardia, tachypnea, an elevated WBC count, and an apparent source of infection.




In [11]:
management = ManagementList([Management("Hospital admission"),
 Management("IV fluids"),
 Management("Ceftriaxone", "1", "g", route="IV", frequency="daily")])

In [12]:
# This is the output to be modified a bit further for investigations (?)
print(json.dumps(ClinicalCase(hopc_parsed, systems, pmh_parsed, pe, primary_ddx, {}, secondary_ddx, secondary_ddx, management).as_dict(), indent=2))

{
  "hopc": {
    "brief_history": "A 26-year-old woman comes to the office because of a 3-day history of lower abdominal pain. She is 18 weeks pregnant by dates.",
    "full_history": "A 26-year-old woman comes to the office because of a 3-day history of lower abdominal pain. She is 18 weeks pregnant by dates. The patient describes the pain as sharp, steady, and radiating across her lower abdomen bilaterally. Last night she developed new nausea and vomiting. She has not been able to keep down any food or drink this morning. She had a normal bowel movement yesterday. She says she felt cold and shivering this morning, followed by feeling warm; however, she did not check her temperature. She denies vaginal bleeding."
  },
  "history": {
    "general": "Patient feels generally weak and ill but was in her usual state of health until 3 days ago. She has gained approximately 5 lbs (2.3 kg) in the pregnancy so far.",
    "skin": "She denies rash.",
    "heent": "Her mouth feels dry. No headac

In [23]:
# NB: This is with some painstaking editing to manually input Investigation findings.
# Need to think of a faster / better way to do it lol
with open("cases/msd_raw/msd_1.json", "r") as f:
    g =json.loads(f.read())
print(ClinicalCase.from_dict(g).prompt())

== HISTORY OF PRESENTING COMPLAINT
BRIEF HISTORY: A 26-year-old woman comes to the office because of a 3-day history of lower abdominal pain. She is 18 weeks pregnant by dates.
FULL HISTORY: A 26-year-old woman comes to the office because of a 3-day history of lower abdominal pain. She is 18 weeks pregnant by dates. The patient describes the pain as sharp, steady, and radiating across her lower abdomen bilaterally. Last night she developed new nausea and vomiting. She has not been able to keep down any food or drink this morning. She had a normal bowel movement yesterday. She says she felt cold and shivering this morning, followed by feeling warm; however, she did not check her temperature. She denies vaginal bleeding.

== SYSTEMS REVIEW
GENERAL: Patient feels generally weak and ill but was in her usual state of health until 3 days ago. She has gained approximately 5 lbs (2.3 kg) in the pregnancy so far.
SKIN: She denies rash.
HEENT: Her mouth feels dry. No headache, nasal congestion, 